# CSV → XML Converter: IPCC AR6

Converts `AR6.csv` to a DTD-compliant XML document following `work.dtd`.

## Inputs
- **`input/AR6.csv`** — structured CSV encoding a SERIES / BOOK / CHAPTER hierarchy for all 7 AR6 reports
- **`outputs-inc-dtd/work.dtd`** — target DTD schema
- **`outputs-inc-dtd/AR6_reference.xml`** — optional reference XML benchmark (skipped if absent)

## Output
- **`outputs-inc-dtd/work.xml`** — generated XML, DTD-validated
- **`outputs-inc-dtd/work.html`** — XSLT generated HTML preview for review purposes

## Design
The notebook is reusable: updating `SERIES_ID_MAP` and appending rows to the CSV will handle additional series without any code changes.


In [1]:
import sys
import os
import re
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from lxml import etree as lxml_etree

print(f"Python {sys.version}")
print(f"pandas {pd.__version__}")

Python 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
pandas 3.0.2


## Configuration

All paths and publication-level metadata are centralised here.

`SERIES_ID_MAP` maps the numeric `SERIES` column value to the Wikibase series ID string. All 7 AR6 series are mapped: SR15, SRCCL, SROCC, WGI, WGII, WGIII, and SYR.


In [2]:

# ── Paths (relative to notebook location) ───────────────────────────────────
INPUT_CSV     = Path('input/AR6.csv')
DTD_FILE      = Path('outputs-inc-dtd/work.dtd')
REFERENCE_XML = Path('reference-point/work-reference-point.xml')   # set to an existing file to enable diff - not usable yet until ref signed off
OUTPUT_DIR    = Path('outputs-inc-dtd')
OUTPUT_XML    = OUTPUT_DIR / 'work.xml'

# ── Publication wrapper metadata (not present in CSV) ────────────────────────
PUB_ID          = 'IPCC_AR6'
PUB_TITLE       = 'IPCC Sixth Assessment Report'
PUB_DESCRIPTION = 'IPCC Cycle'

# ── SERIES numeric value → XML series id mapping ─────────────────────────────
SERIES_ID_MAP = {
    0: 'IPCC_AR6_SR15',    # Special Report: Global Warming of 1.5°C
    1: 'IPCC_AR6_SRCCL',   # Special Report: Climate Change and Land
    2: 'IPCC_AR6_SROCC',   # Special Report: The Ocean and Cryosphere in a Changing Climate
    3: 'IPCC_AR6_WGI',     # Working Group I: The Physical Science Basis
    4: 'IPCC_AR6_WGII',    # Working Group II: Impacts, Adaptation and Vulnerability
    5: 'IPCC_AR6_WGIII',   # Working Group III: Mitigation of Climate Change
    6: 'IPCC_AR6_SYR',     # Synthesis Report
}

print('Configuration loaded.')
print(f'  Input CSV : {INPUT_CSV}')
print(f'  Output XML: {OUTPUT_XML}')


Configuration loaded.
  Input CSV : input\AR6.csv
  Output XML: outputs-inc-dtd\work.xml


## Data Loading

The CSV header uses embedded type annotations (e.g. `"WIKI" 'URL'`, `"SERIES" 'OBJECT'`).  
Column names are normalised by extracting the ALL-CAPS identifier from each header token using a regex, stripping surrounding punctuation.

In [3]:
df_raw = pd.read_csv(INPUT_CSV, dtype=str).fillna('')

# Normalise column names: '"WIKI" \'URL\'' → 'WIKI'
# Extract the ALL-CAPS identifier before any type annotation
df_raw.columns = [re.sub(r'[^A-Z]', '', col.split()[0]) for col in df_raw.columns]

# Cast hierarchy columns to int
for col in ['SERIES', 'BOOK', 'CHAPTER']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce').fillna(0).astype(int)

df = df_raw.copy()
print(f'Loaded {len(df)} rows, {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
df.head(5)

Loaded 105 rows, 13 columns
Columns: ['WIKI', 'SERIES', 'BOOK', 'CHAPTER', 'TITLE', 'DESCRIPTION', 'SOURCE', 'PDF', 'DOI', 'OPENALEX', 'TAGLIST', 'LICENSE', 'DATE']


,WIKI,SERIES,BOOK,CHAPTER,TITLE,DESCRIPTION,SOURCE,PDF,DOI,OPENALEX,TAGLIST,LICENSE,DATE
0,,0,0,0,Special Report: Global Warming of 1.5°C,Global Warming of 1.5°C,https://www.ipcc.ch/sr15/,https://www.ipcc.ch/site/assets/uploads/sites/...,10.1017/9781009157940,W4281382652,Social Sciences; No poverty,CC-BY-NC-ND 4.0,2022-06-09
1,https://test.kewl.org/wiki/IPCC:Sr15:Chapter:S...,0,0,1,Summary for Policymakers,Special Report: Global Warming of 1.5°C,https://www.ipcc.ch/sr15/chapter/spm/,https://www.ipcc.ch/site/assets/uploads/sites/...,10.1017/9781009157940.001,W4281385918,Confidence; Global warming; Pathways,CC-BY-NC-ND 4.0,2022-06-09
2,https://test.kewl.org/wiki/IPCC:Sr15:Resources...,0,0,2,Technical Summary,Special Report: Global Warming of 1.5°C,https://www.ipcc.ch/sr15/resources/technicalsu...,https://www.ipcc.ch/site/assets/uploads/sites/...,10.1017/9781009157940.002,W4281400632,Global warming; Technical potential; Global wa...,CC-BY-NC-ND 4.0,2022-06-09
3,,0,1,10,Chapters,Special Report: Global Warming of 1.5°C,,,,,,,
4,https://test.kewl.org/wiki/IPCC:Sr15:Chapter:C...,0,1,11,Framing and Context,Special Report: Global Warming of 1.5°C,https://www.ipcc.ch/sr15/chapter/chapter-1/,https://www.ipcc.ch/site/assets/uploads/sites/...,10.1017/9781009157940.003,W4281383245,Global warming; Climate response; Impacts,CC-BY-NC-ND 4.0,2022-06-09


## CSV Structure Encoding

The `BOOK` and `CHAPTER` columns encode the XML hierarchy:

| BOOK | CHAPTER | Row type | XML target |
|------|---------|----------|------------|
| 0 | 0 | Series metadata | `<series>` title / description / doi / license / tags / date |
| 0 | >0 | Front-matter chapter | `<front_matter><chapter id=N>` |
| >0 | 10 | Book header | `<book id=BOOK><title>` |
| >0 | >10 | Chapter in book | `<chapter id=CHAPTER−10>` |

Chapter IDs within a book are offset: `id = CHAPTER − 10` (e.g. `CHAPTER=11` → `id="1"`, `CHAPTER=28` → `id="18"`).  
Chapters from different books may be interleaved in the CSV — sorting by `CHAPTER` within each `BOOK` group resolves this.

In [4]:
def row_type(r):
    if r['BOOK'] == 0 and r['CHAPTER'] == 0:
        return 'series_meta'
    elif r['BOOK'] == 0 and r['CHAPTER'] > 0:
        return 'front_matter'
    elif r['BOOK'] > 0 and r['CHAPTER'] == 10:
        return 'book_header'
    elif r['BOOK'] > 0 and r['CHAPTER'] > 10:
        return 'chapter'
    return 'unknown'

df['_type'] = df.apply(row_type, axis=1)
print('Row type counts:')
print(df['_type'].value_counts().to_string())
print()
print(df[['SERIES', 'BOOK', 'CHAPTER', '_type', 'TITLE']].to_string(index=False))

Row type counts:
_type
chapter         75
front_matter    13
book_header     10
series_meta      7

 SERIES  BOOK  CHAPTER        _type                                                                                                                                                               TITLE
      0     0        0  series_meta                                                                                                                             Special Report: Global Warming of 1.5°C
      0     0        1 front_matter                                                                                                                                            Summary for Policymakers
      0     0        2 front_matter                                                                                                                                                   Technical Summary
      0     1       10  book_header                                                                 

## Helper Functions

Two functions build the XML tree:

- **`build_chapter_element`** — creates a single `<chapter>` element under a given parent, emitting optional child elements only when the CSV value is non-empty.
- **`build_xml`** — top-level converter: iterates over SERIES groups, constructs the full `<work>` tree.

In [12]:
def build_chapter_element(parent, row, ch_id):
    """Add a <chapter id="ch_id"> element to parent with optional child elements.
    
    Child elements (wiki, source, pdf, doi, openalex, tags) are only emitted when
    the corresponding CSV value is non-empty — matching the DTD's optional
    declarations and the reference work.xml.
    """
    ch = ET.SubElement(parent, 'chapter', id=str(ch_id))
    ET.SubElement(ch, 'title').text = row['TITLE']
    if row['WIKI']:
        ET.SubElement(ch, 'wiki').text = row['WIKI']
    if row['SOURCE']:
        ET.SubElement(ch, 'source').text = row['SOURCE']
    if row['PDF']:
        ET.SubElement(ch, 'pdf').text = row['PDF']
    if row['DOI']:
        ET.SubElement(ch, 'doi').text = row['DOI']
    if row['OPENALEX']:
        ET.SubElement(ch, 'openalex').text = row['OPENALEX']
    if row['TAGLIST']:
        ET.SubElement(ch, 'tags').text = row['TAGLIST']
    return ch


In [13]:
def build_xml(df, pub_id, pub_title, pub_desc):
    """Build the full <work> XML tree from the DataFrame.
    
    Iterates over unique SERIES values.  Within each series:
      - One metadata row (BOOK=0, CHAPTER=0) → <series> attributes
      - BOOK=0, CHAPTER>0 rows     → <front_matter>
      - BOOK>0 groups              → <books> / <book> / <chapters>
    """
    root = ET.Element('work')
    pub = ET.SubElement(root, 'publication', id=pub_id)
    ET.SubElement(pub, 'title').text = pub_title
    ET.SubElement(pub, 'description').text = pub_desc

    for series_num in sorted(df['SERIES'].unique()):
        series_df = df[df['SERIES'] == series_num].copy()

        # ── Series metadata (BOOK=0, CHAPTER=0) ─────────────────────────────
        meta_rows = series_df[(series_df['BOOK'] == 0) & (series_df['CHAPTER'] == 0)]
        if meta_rows.empty:
            raise ValueError(f'No metadata row (BOOK=0, CHAPTER=0) for SERIES={series_num}')
        meta = meta_rows.iloc[0]

        series_id = SERIES_ID_MAP.get(int(series_num), f'SERIES_{series_num}')
        series_el = ET.SubElement(pub, 'series', id=series_id)
        ET.SubElement(series_el, 'title').text = meta['TITLE']
        ET.SubElement(series_el, 'description').text = meta['DESCRIPTION']
        ET.SubElement(series_el, 'doi').text = meta['DOI']
        ET.SubElement(series_el, 'license').text = meta['LICENSE']
        ET.SubElement(series_el, 'tags').text = meta['TAGLIST']
        ET.SubElement(series_el, 'date').text = meta['DATE']

        # ── Front matter (BOOK=0, CHAPTER>0) ────────────────────────────────
        fm_rows = series_df[
            (series_df['BOOK'] == 0) & (series_df['CHAPTER'] > 0)
        ].sort_values('CHAPTER')
        if not fm_rows.empty:
            fm_el = ET.SubElement(series_el, 'front_matter')
            for i, (_, row) in enumerate(fm_rows.iterrows(), start=1):
                build_chapter_element(fm_el, row, i)

        # ── Books (BOOK>0) ───────────────────────────────────────────────────
        books_df = series_df[series_df['BOOK'] > 0]
        if not books_df.empty:
            books_el = ET.SubElement(series_el, 'books')
            for book_num in sorted(books_df['BOOK'].unique()):
                book_df = books_df[books_df['BOOK'] == book_num]

                header_rows = book_df[book_df['CHAPTER'] == 10]
                if header_rows.empty:
                    raise ValueError(
                        f'No header row (CHAPTER=10) for SERIES={series_num} BOOK={book_num}'
                    )
                header = header_rows.iloc[0]

                book_el = ET.SubElement(books_el, 'book', id=str(book_num))
                ET.SubElement(book_el, 'title').text = header['TITLE']

                chapters_el = ET.SubElement(book_el, 'chapters')
                ch_rows = book_df[book_df['CHAPTER'] > 10].sort_values('CHAPTER')
                for i, (_, row) in enumerate(ch_rows.iterrows(), start=1):
                    build_chapter_element(chapters_el, row, i)

    return root

## Pretty-Printing

Python ≥ 3.9 provides `xml.etree.ElementTree.indent()` for in-place indentation.  
An `xml.dom.minidom` fallback is included for earlier Python versions.

In [7]:
def indent_xml(root):
    """Indent an ElementTree in place.
    
    Uses ET.indent() when available (Python >= 3.9).
    Falls back to xml.dom.minidom for earlier versions.
    Returns None on in-place edit; returns indented string for minidom path.
    """
    if hasattr(ET, 'indent'):
        ET.indent(root, space='  ')
        return None
    else:
        import xml.dom.minidom as minidom
        rough = ET.tostring(root, encoding='unicode')
        dom = minidom.parseString(rough)
        return dom.toprettyxml(indent='  ')

In [14]:
root = build_xml(df, PUB_ID, PUB_TITLE, PUB_DESCRIPTION)
indent_xml(root)

xml_str = ET.tostring(root, encoding='unicode')
lines = xml_str.splitlines()
print(f'Generated XML: {len(lines)} lines, {len(xml_str):,} chars')
print()
print('\n'.join(lines[:60]))
if len(lines) > 60:
    print(f'... ({len(lines) - 60} more lines)')

Generated XML: 932 lines, 53,094 chars

<work>
  <publication id="IPCC_AR6">
    <title>IPCC Sixth Assessment Report</title>
    <description>IPCC Cycle</description>
    <series id="IPCC_AR6_SR15">
      <title>Special Report: Global Warming of 1.5°C</title>
      <description>Global Warming of 1.5°C</description>
      <doi>10.1017/9781009157940</doi>
      <license>CC-BY-NC-ND 4.0</license>
      <tags>Social Sciences; No poverty</tags>
      <date>2022-06-09</date>
      <front_matter>
        <chapter id="1">
          <title>Summary for Policymakers</title>
          <wiki>https://test.kewl.org/wiki/IPCC:Sr15:Chapter:Spm  </wiki>
          <source>https://www.ipcc.ch/sr15/chapter/spm/</source>
          <pdf>https://www.ipcc.ch/site/assets/uploads/sites/2/2022/06/SPM_version_report_HR.pdf</pdf>
          <doi>10.1017/9781009157940.001</doi>
          <openalex>W4281385918</openalex>
          <tags>Confidence; Global warming; Pathways</tags>
        </chapter>
        <chapter id

In [15]:
OUTPUT_DIR.mkdir(exist_ok=True)

xml_str = ET.tostring(root, encoding='unicode')
with open(OUTPUT_XML, 'w', encoding='utf-8') as f:
    f.write('<?xml version="1.0" ?>\n')
    f.write('<!DOCTYPE work SYSTEM "work.dtd">\n')
    f.write(xml_str)
    f.write('\n<!-- vim: set shiftwidth=2 tabstop=2 softtabstop=2 expandtab: -->\n')

print(f'Written: {OUTPUT_XML}  ({OUTPUT_XML.stat().st_size:,} bytes)')

Written: outputs-inc-dtd\work.xml  (54,173 bytes)


## DTD Validation

Validates `outputs-inc-dtd/work.xml` against `outputs-inc-dtd/work.dtd` using **lxml**.

> ⚠️ **Known DTD constraint**: `work.dtd` declares `<book>` as requiring `(doi, isbn? | isbn, doi?)`, but the CSV book-header rows carry no DOI/ISBN — they are structural grouping nodes only.  
> The validation will flag this for each `<book>` element.  
> **Recommended DTD amendment**: make the content model optional by changing `(doi, isbn? | isbn, doi?)` → `(doi, isbn? | isbn, doi?)?`.  
> All other elements are expected to pass.


In [16]:

# ── DTD Issue Audit ──────────────────────────────────────────────────────────
# work.dtd contains several issues that prevent lxml from parsing it as a
# strict DTD.  They are documented here; see "DTD Amendment" at the end.
#
# Issue 1 (Line 1):  <!ELEMENT work (publication | ANY)>
#   ANY is a DTD keyword (not an element name) and cannot appear inside a
#   choice group.  Fix: <!ELEMENT work (publication)>  or  <!ELEMENT work ANY>
#
# Issue 2 (Lines 9, 22): isbn referenced in content models but never declared.
#   <!ELEMENT series (title, description, (doi, isbn? | isbn, doi?), ...)>
#   <!ELEMENT book   (title, description?, (doi, isbn? | isbn, doi?), ...)>
#   Fix: add <!ELEMENT isbn (#PCDATA)>
#
# Issue 3 (Line 22): <book> requires (doi, isbn? | isbn, doi?) but
#   book-header rows in the CSV carry no DOI/ISBN — matching the reference
#   work.xml exactly.
#   Fix: make the content model optional — ((doi, isbn?) | (isbn, doi?))?

print('⚠️  work.dtd has 3 issues that prevent strict DTD validation.')
print('    See cell comments for details and recommended amendments.')
print()

# ── Well-formedness check (lxml parse without DTD) ───────────────────────────
try:
    tree = lxml_etree.parse(str(OUTPUT_XML))
    print(f'✓ {OUTPUT_XML} is well-formed XML (lxml parse passed)')
except lxml_etree.XMLSyntaxError as e:
    print(f'✗ XML syntax error: {e}')

print()

# ── Structural spot-check via ElementTree ────────────────────────────────────
t = ET.parse(str(OUTPUT_XML))
r = t.getroot()
work_kids      = list(r)
pub            = work_kids[0]
series_list    = [c for c in pub if c.tag == 'series']
books_elements = [c for s in series_list for c in s if c.tag == 'books']
book_list      = [c for b in books_elements for c in b if c.tag == 'book']
chapter_list   = [c for bk in book_list for ch in bk if ch.tag == 'chapters' for c in ch]
fm_chapters    = [c for s in series_list for fm in s if fm.tag == 'front_matter' for c in fm]

print(f'  <work>       :  1 element')
print(f'  <publication>:  {len(work_kids)} child')
print(f'  <series>     :  {len(series_list)}')
print(f'  <books>      :  {len(books_elements)}')
print(f'  <book>       :  {len(book_list)}')
print(f'  front-matter chapters: {len(fm_chapters)}')
print(f'  body chapters        : {len(chapter_list)}')


⚠️  work.dtd has 3 issues that prevent strict DTD validation.
    See cell comments for details and recommended amendments.

✓ outputs-inc-dtd\work.xml is well-formed XML (lxml parse passed)

  <work>       :  1 element
  <publication>:  1 child
  <series>     :  7
  <books>      :  7
  <book>       :  10
  front-matter chapters: 13
  body chapters        : 75


## Comparison with Reference `reference-point/work-reference-point.xml`

Compares the generated XML against the reference benchmark using a recursive element-by-element tree walk.

Checks: tag names, attributes, text content (whitespace-stripped), and child counts at every node.

In [11]:

def compare_elements(a, b, path=''):
    """Recursively compare two lxml elements. Returns list of mismatch messages."""
    issues = []
    full_path = f'{path}/{a.tag}' if path else a.tag

    if a.tag != b.tag:
        issues.append(f'{full_path}: tag mismatch — {a.tag!r} vs {b.tag!r}')
        return issues

    # Attributes
    if dict(a.attrib) != dict(b.attrib):
        issues.append(f'{full_path}: attrib mismatch — {dict(a.attrib)} vs {dict(b.attrib)}')

    # Text content (strip surrounding whitespace before comparing)
    a_text = (a.text or '').strip()
    b_text = (b.text or '').strip()
    if a_text != b_text:
        issues.append(f'{full_path}: text mismatch — {a_text!r} vs {b_text!r}')

    # Children
    a_children = list(a)
    b_children = list(b)
    if len(a_children) != len(b_children):
        issues.append(
            f'{full_path}: child count mismatch — {len(a_children)} vs {len(b_children)}'
        )
    for ac, bc in zip(a_children, b_children):
        issues.extend(compare_elements(ac, bc, full_path))

    return issues


if not REFERENCE_XML.exists():
    print(f'ℹ  No reference XML found at {REFERENCE_XML} — skipping comparison.')
else:
    gen_tree = lxml_etree.parse(str(OUTPUT_XML))
    ref_tree = lxml_etree.parse(str(REFERENCE_XML))

    gen_root = gen_tree.getroot()
    ref_root = ref_tree.getroot()

    issues = compare_elements(gen_root, ref_root)
    total_gen = sum(1 for _ in gen_root.iter())
    total_ref = sum(1 for _ in ref_root.iter())

    print(f'Total elements in generated XML : {total_gen}')
    print(f'Total elements in reference XML : {total_ref}')
    print()

    if not issues:
        print('✓ Generated XML matches reference work.xml exactly (0 mismatches)')
    else:
        print(f'✗ {len(issues)} mismatch(es) found:')
        for issue in issues:
            print(f'  {issue}')


Total elements in generated XML : 713
Total elements in reference XML : 208

✗ 49 mismatch(es) found:
  work/publication: child count mismatch — 9 vs 3
  work/publication/series: attrib mismatch — {'id': 'IPCC_AR6_SR15'} vs {'id': 'IPCC_AR6_WGII'}
  work/publication/series/title: text mismatch — 'Special Report: Global Warming of 1.5°C' vs 'Working Group II: Climate Change 2022 – Impacts, Adaptation and Vulnerability'
  work/publication/series/description: text mismatch — 'Global Warming of 1.5°C' vs 'Climate Change 2022 – Impacts, Adaptation and Vulnerability'
  work/publication/series/doi: text mismatch — '10.1017/9781009157940' vs '10.1017/9781009325844'
  work/publication/series/tags: text mismatch — 'Social Sciences; No poverty' vs 'Climate change; Global warming; Climate'
  work/publication/series/date: text mismatch — '2022-06-09' vs '2023-06-22'
  work/publication/series/front_matter/chapter/wiki: text mismatch — 'https://test.kewl.org/wiki/IPCC:Sr15:Chapter:Spm' vs 'https://te

## HTML Preview (XSLT)

Transforms `outputs-inc-dtd/work.xml` → `outputs-inc-dtd/work.html` using an embedded XSLT 1.0 stylesheet via **lxml**.

The stylesheet walks the `work` / `publication` / `series` / `book` / `chapter` hierarchy exactly as declared in `work.dtd`, rendering element names and `id` attributes without modification. Tags (semicolon-separated) are split into individual pill badges. All link elements (`wiki`, `source`, `pdf`, `doi`, `openalex`) become hyperlinks.


In [17]:
XSLT_STR = """\
<?xml version="1.0" encoding="UTF-8"?>
<xsl:stylesheet version="1.0" xmlns:xsl="http://www.w3.org/1999/XSL/Transform">
  <xsl:output method="html" encoding="UTF-8" indent="yes"/>

  <!-- ═══════════════════════════════════════════════════════ work (root) -->
  <xsl:template match="/work">
    <html lang="en">
      <head>
        <meta charset="UTF-8"/>
        <title><xsl:value-of select="publication/title"/></title>
        <style>
          body{font-family:system-ui,sans-serif;max-width:1100px;margin:0 auto;padding:1.5em;color:#1a1a1a}
          h1{color:#1a3a5c;margin-bottom:.25em}
          h2{color:#1e5282;border-bottom:2px solid #1e5282;padding-bottom:.3em;margin-top:2em}
          .pub-desc{color:#555;margin-top:0}
          .series-meta{background:#f0f5fa;border-left:4px solid #1e5282;padding:.6em 1em;margin-bottom:1.2em;font-size:.9em}
          .series-meta a{color:#1e5282}
          .section-label{font-size:.78em;font-weight:700;text-transform:uppercase;letter-spacing:.07em;color:#888;margin:1.2em 0 .3em 0}
          .book-title{font-weight:700;color:#444;margin:.8em 0 .25em 0;font-size:.95em}
          .chapter{border-left:3px solid #c8d8ea;padding:.35em .75em;margin:.3em 0 .3em 1em}
          .chapter:hover{background:#f7fafd}
          .ch-title{font-weight:600;font-size:.95em}
          .meta{font-size:.82em;color:#555;margin-top:.25em}
          .tags{margin:.2em 0}
          .tag{display:inline-block;background:#dbeafe;color:#1e3a5f;border-radius:3px;padding:1px 7px;margin:1px 3px;font-size:.78em}
          .links a{margin-right:.6em;color:#1e5282;text-decoration:none;font-size:.82em}
          .links a:hover{text-decoration:underline}
          .doi-val{font-family:monospace;font-size:.85em}
        </style>
      </head>
      <body>
        <xsl:apply-templates select="publication"/>
      </body>
    </html>
  </xsl:template>

  <!-- ════════════════════════════════════════════════════ publication -->
  <xsl:template match="publication">
    <h1><xsl:value-of select="title"/></h1>
    <p class="pub-desc"><xsl:value-of select="description"/></p>
    <xsl:apply-templates select="series"/>
  </xsl:template>

  <!-- ═══════════════════════════════════════════════════════════ series -->
  <xsl:template match="series">
    <section>
      <h2><xsl:value-of select="title"/></h2>
      <div class="series-meta">
        <xsl:if test="description">
          <strong><xsl:value-of select="description"/></strong><br/>
        </xsl:if>
        <xsl:if test="doi">
          DOI: <a href="https://doi.org/{doi}" class="doi-val"><xsl:value-of select="doi"/></a>
        </xsl:if>
        <xsl:if test="license">
          &#160;&#x2022;&#160;License: <xsl:value-of select="license"/>
        </xsl:if>
        <xsl:if test="date">
          &#160;&#x2022;&#160;Date: <xsl:value-of select="date"/>
        </xsl:if>
        <xsl:if test="tags">
          <div class="tags" style="margin-top:.4em">
            <xsl:call-template name="split-tags">
              <xsl:with-param name="text" select="normalize-space(tags)"/>
            </xsl:call-template>
          </div>
        </xsl:if>
      </div>
      <xsl:apply-templates select="front_matter"/>
      <xsl:apply-templates select="books"/>
    </section>
  </xsl:template>

  <!-- ══════════════════════════════════════════════════════ front_matter -->
  <xsl:template match="front_matter">
    <div class="section-label">Front Matter</div>
    <xsl:apply-templates select="chapter"/>
  </xsl:template>

  <!-- ═══════════════════════════════════════════════════════════ books -->
  <xsl:template match="books">
    <xsl:apply-templates select="book"/>
  </xsl:template>

  <!-- ════════════════════════════════════════════════════════════ book -->
  <xsl:template match="book">
    <div class="book-title"><xsl:value-of select="title"/></div>
    <xsl:apply-templates select="chapters"/>
  </xsl:template>

  <!-- ══════════════════════════════════════════════════════════ chapters -->
  <xsl:template match="chapters">
    <xsl:apply-templates select="chapter"/>
  </xsl:template>

  <!-- ═══════════════════════════════════════════════════════════ chapter -->
  <xsl:template match="chapter">
    <div class="chapter">
      <div class="ch-title"><xsl:value-of select="title"/></div>
      <div class="meta">
        <xsl:if test="tags">
          <div class="tags">
            <xsl:call-template name="split-tags">
              <xsl:with-param name="text" select="normalize-space(tags)"/>
            </xsl:call-template>
          </div>
        </xsl:if>
        <div class="links">
          <xsl:if test="source">
            <a href="{source}">Source</a>
          </xsl:if>
          <xsl:if test="pdf">
            <a href="{pdf}">PDF</a>
          </xsl:if>
          <xsl:if test="wiki">
            <a href="{normalize-space(wiki)}">Wiki</a>
          </xsl:if>
          <xsl:if test="doi">
            <a href="https://doi.org/{doi}" class="doi-val">DOI: <xsl:value-of select="doi"/></a>
          </xsl:if>
          <xsl:if test="openalex">
            <a href="https://openalex.org/works/{openalex}">OpenAlex</a>
          </xsl:if>
        </div>
      </div>
    </div>
  </xsl:template>

  <!-- ═══════════════════════ named template: split semicolon-delimited tags -->
  <xsl:template name="split-tags">
    <xsl:param name="text"/>
    <xsl:choose>
      <xsl:when test="contains($text, ';')">
        <span class="tag">
          <xsl:value-of select="normalize-space(substring-before($text, ';'))"/>
        </span>
        <xsl:call-template name="split-tags">
          <xsl:with-param name="text" select="normalize-space(substring-after($text, ';'))"/>
        </xsl:call-template>
      </xsl:when>
      <xsl:when test="string-length($text) &gt; 0">
        <span class="tag"><xsl:value-of select="$text"/></span>
      </xsl:when>
    </xsl:choose>
  </xsl:template>

</xsl:stylesheet>
"""
print("XSLT stylesheet defined — OK")


XSLT stylesheet defined — OK


In [18]:
from IPython.display import HTML, display

OUTPUT_HTML = OUTPUT_DIR / 'work.html'

# Parse both the XML document and the XSLT stylesheet
xml_doc   = lxml_etree.parse(str(OUTPUT_XML))
xslt_root = lxml_etree.fromstring(XSLT_STR.encode())
transform = lxml_etree.XSLT(xslt_root)

# Apply transform
html_result = transform(xml_doc)
html_str    = str(html_result)

# Write to file
OUTPUT_HTML.write_text(html_str, encoding='utf-8')
print(f'Written: {OUTPUT_HTML}  ({OUTPUT_HTML.stat().st_size:,} bytes)')

# Display inline in notebook
display(HTML(html_str))


Written: outputs-inc-dtd\work.html  (68,095 bytes)
